# FBRef Paste Parser

Paste schedule data from FBRef directly into the `RAW_TEXT` variable below.
Run the parse cell and it appends to the master CSV.

**Format expected:** tab-separated rows copied from FBRef schedule pages:
`Wk  Day  Date  Time  Home  Score  Away  Attendance  Venue  Referee  Match Report  Notes`

**What to paste for each season (5 competitions × 5 seasons = ~25 pastes):**
- Premier League
- FA Cup
- League Cup
- UEFA Champions League
- UEFA Europa League
- UEFA Europa Conference League (from 2021-22 onwards)

In [1]:
import pandas as pd
import re
from pathlib import Path
from io import StringIO

OUTPUT_DIR = Path("../../../../infra/data/json/fbref").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_CSV = OUTPUT_DIR / "fixtures_manual.csv"

print(f"Master CSV: {MASTER_CSV}")
if MASTER_CSV.exists():
    existing = pd.read_csv(MASTER_CSV)
    print(f"Existing rows: {len(existing)}")
else:
    print("No file yet — will be created on first parse")

Master CSV: /Users/admin/dev/algobetting/infra/data/json/fbref/fixtures_manual.csv
No file yet — will be created on first parse


## Paste cell

1. Set `COMPETITION` and `SEASON`
2. Paste the copied FBRef table into `RAW_TEXT`
3. Run this cell and the one below

In [2]:
COMPETITION = "Premier League"   # change for each paste
SEASON      = "2017-2018"        # change for each paste

RAW_TEXT = """

Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
1	Fri	2017-08-11	19:45 (20:45)	Arsenal	4–3	Leicester City	59,387	Emirates Stadium	Mike Dean	Match Report	
1	Sat	2017-08-12	12:30 (13:30)	Watford	3–3	Liverpool	20,407	Vicarage Road Stadium	Anthony Taylor	Match Report	
1	Sat	2017-08-12	15:00 (16:00)	Everton	1–0	Stoke City	39,045	Goodison Park	Niel Swarbrick	Match Report	
1	Sat	2017-08-12	15:00 (16:00)	Chelsea	2–3	Burnley	41,616	Stamford Bridge	Craig Pawson	Match Report	
1	Sat	2017-08-12	15:00 (16:00)	Crystal Palace	0–3	Huddersfield Town	25,448	Selhurst Park	Jonathan Moss	Match Report	
1	Sat	2017-08-12	15:00 (16:00)	West Brom	1–0	Bournemouth	25,011	The Hawthorns	Robert Madley	Match Report	
1	Sat	2017-08-12	15:00 (16:00)	Southampton	0–0	Swansea City	31,447	St. Mary's Stadium	Mike Jones	Match Report	
1	Sat	2017-08-12	17:30 (18:30)	Brighton	0–2	Manchester City	30,415	The American Express Community Stadium	Michael Oliver	Match Report	
1	Sun	2017-08-13	13:30 (14:30)	Newcastle United	0–2	Tottenham Hotspur	52,077	St. James' Park	Andre Marriner	Match Report	
1	Sun	2017-08-13	16:00 (17:00)	Manchester Utd	4–0	West Ham United	74,928	Old Trafford	Martin Atkinson	Match Report	
2	Sat	2017-08-19	12:30 (13:30)	Swansea City	0–4	Manchester Utd	20,862	Liberty Stadium	Jonathan Moss	Match Report	
2	Sat	2017-08-19	15:00 (16:00)	Bournemouth	0–2	Watford	10,501	Vitality Stadium	Roger East	Match Report	
2	Sat	2017-08-19	15:00 (16:00)	Burnley	0–1	West Brom	19,619	Turf Moor	Martin Atkinson	Match Report	
2	Sat	2017-08-19	15:00 (16:00)	Southampton	3–2	West Ham United	31,424	St. Mary's Stadium	Lee Mason	Match Report	
2	Sat	2017-08-19	15:00 (16:00)	Leicester City	2–0	Brighton	31,902	King Power Stadium	Lee Probert	Match Report	
2	Sat	2017-08-19	15:00 (16:00)	Liverpool	1–0	Crystal Palace	53,138	Anfield	Kevin Friend	Match Report	
2	Sat	2017-08-19	17:30 (18:30)	Stoke City	1–0	Arsenal	29,459	Bet365 Stadium	Andre Marriner	Match Report	
2	Sun	2017-08-20	13:30 (14:30)	Huddersfield Town	1–0	Newcastle United	24,128	The John Smith's Stadium	Craig Pawson	Match Report	
2	Sun	2017-08-20	16:00 (17:00)	Tottenham Hotspur	1–2	Chelsea	73,587	Wembley Stadium	Anthony Taylor	Match Report	
2	Mon	2017-08-21	20:00 (21:00)	Manchester City	1–1	Everton	49,108	Etihad Stadium	Robert Madley	Match Report	
3	Sat	2017-08-26	12:30 (13:30)	Bournemouth	1–2	Manchester City	10,419	Vitality Stadium	Mike Dean	Match Report	
3	Sat	2017-08-26	15:00 (16:00)	Newcastle United	3–0	West Ham United	52,093	St. James' Park	Niel Swarbrick	Match Report	
3	Sat	2017-08-26	15:00 (16:00)	Watford	0–0	Brighton	20,181	Vicarage Road Stadium	Graham Scott	Match Report	
3	Sat	2017-08-26	15:00 (16:00)	Huddersfield Town	0–0	Southampton	23,548	The John Smith's Stadium	Stuart Attwell	Match Report	
3	Sat	2017-08-26	15:00 (16:00)	Crystal Palace	0–2	Swansea City	23,477	Selhurst Park	Andre Marriner	Match Report	
3	Sat	2017-08-26	17:30 (18:30)	Manchester Utd	2–0	Leicester City	75,021	Old Trafford	Michael Oliver	Match Report	
3	Sun	2017-08-27	13:30 (14:30)	West Brom	1–1	Stoke City	22,704	The Hawthorns	Anthony Taylor	Match Report	
3	Sun	2017-08-27	13:30 (14:30)	Chelsea	2–0	Everton	41,382	Stamford Bridge	Jonathan Moss	Match Report	
3	Sun	2017-08-27	16:00 (17:00)	Tottenham Hotspur	1–1	Burnley	67,862	Wembley Stadium	Lee Mason	Match Report	
3	Sun	2017-08-27	16:00 (17:00)	Liverpool	4–0	Arsenal	53,206	Anfield	Craig Pawson	Match Report	
4	Sat	2017-09-09	12:30 (13:30)	Manchester City	5–0	Liverpool	54,172	Etihad Stadium	Jonathan Moss	Match Report	
4	Sat	2017-09-09	15:00 (16:00)	Arsenal	3–0	Bournemouth	59,262	Emirates Stadium	Anthony Taylor	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
4	Sat	2017-09-09	15:00 (16:00)	Southampton	0–2	Watford	31,435	St. Mary's Stadium	Lee Probert	Match Report	
4	Sat	2017-09-09	15:00 (16:00)	Leicester City	1–2	Chelsea	30,923	King Power Stadium	Lee Mason	Match Report	
4	Sat	2017-09-09	15:00 (16:00)	Brighton	3–1	West Brom	30,381	The American Express Community Stadium	Chris Kavanagh	Match Report	
4	Sat	2017-09-09	15:00 (16:00)	Everton	0–3	Tottenham Hotspur	38,835	Goodison Park	Graham Scott	Match Report	
4	Sat	2017-09-09	17:30 (18:30)	Stoke City	2–2	Manchester Utd	29,320	Bet365 Stadium	Niel Swarbrick	Match Report	
4	Sun	2017-09-10	13:30 (14:30)	Burnley	1–0	Crystal Palace	18,862	Turf Moor	Michael Oliver	Match Report	
4	Sun	2017-09-10	16:00 (17:00)	Swansea City	0–1	Newcastle United	20,872	Liberty Stadium	Mike Jones	Match Report	
4	Mon	2017-09-11	20:00 (21:00)	West Ham United	2–0	Huddersfield Town	56,977	London Stadium	Kevin Friend	Match Report	
5	Fri	2017-09-15	20:00 (21:00)	Bournemouth	2–1	Brighton	10,369	Vitality Stadium	Craig Pawson	Match Report	
5	Sat	2017-09-16	12:30 (13:30)	Crystal Palace	0–1	Southampton	24,199	Selhurst Park	Robert Madley	Match Report	
5	Sat	2017-09-16	15:00 (16:00)	Newcastle United	2–1	Stoke City	51,795	St. James' Park	Stuart Attwell	Match Report	
5	Sat	2017-09-16	15:00 (16:00)	Liverpool	1–1	Burnley	53,231	Anfield	Roger East	Match Report	
5	Sat	2017-09-16	15:00 (16:00)	Watford	0–6	Manchester City	20,305	Vicarage Road Stadium	Anthony Taylor	Match Report	
5	Sat	2017-09-16	15:00 (16:00)	West Brom	0–0	West Ham United	24,942	The Hawthorns	Paul Tierney	Match Report	
5	Sat	2017-09-16	15:00 (16:00)	Huddersfield Town	1–1	Leicester City	24,129	The John Smith's Stadium	Jonathan Moss	Match Report	
5	Sat	2017-09-16	17:30 (18:30)	Tottenham Hotspur	0–0	Swansea City	65,366	Wembley Stadium	Mike Dean	Match Report	
5	Sun	2017-09-17	13:30 (14:30)	Chelsea	0–0	Arsenal	41,478	Stamford Bridge	Michael Oliver	Match Report	
5	Sun	2017-09-17	16:00 (17:00)	Manchester Utd	4–0	Everton	75,042	Old Trafford	Andre Marriner	Match Report	
6	Sat	2017-09-23	12:30 (13:30)	West Ham United	2–3	Tottenham Hotspur	56,988	London Stadium	Michael Oliver	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Everton	2–1	Bournemouth	38,133	Goodison Park	Martin Atkinson	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Southampton	0–1	Manchester Utd	31,930	St. Mary's Stadium	Craig Pawson	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Stoke City	0–4	Chelsea	29,661	Bet365 Stadium	Mike Dean	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Burnley	0–0	Huddersfield Town	20,759	Turf Moor	Chris Kavanagh	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Manchester City	5–0	Crystal Palace	53,526	Etihad Stadium	Niel Swarbrick	Match Report	
6	Sat	2017-09-23	15:00 (16:00)	Swansea City	1–2	Watford	20,372	Liberty Stadium	Lee Mason	Match Report	
6	Sat	2017-09-23	17:30 (18:30)	Leicester City	2–3	Liverpool	32,004	King Power Stadium	Anthony Taylor	Match Report	
6	Sun	2017-09-24	16:00 (17:00)	Brighton	1–0	Newcastle United	30,468	The American Express Community Stadium	Andre Marriner	Match Report	
6	Mon	2017-09-25	20:00 (21:00)	Arsenal	2–0	West Brom	59,134	Emirates Stadium	Robert Madley	Match Report	
7	Sat	2017-09-30	12:30 (13:30)	Huddersfield Town	0–4	Tottenham Hotspur	24,169	The John Smith's Stadium	Niel Swarbrick	Match Report	
7	Sat	2017-09-30	15:00 (16:00)	Manchester Utd	4–0	Crystal Palace	75,118	Old Trafford	Mike Dean	Match Report	
7	Sat	2017-09-30	15:00 (16:00)	West Ham United	1–0	Swansea City	56,922	London Stadium	Roger East	Match Report	
7	Sat	2017-09-30	15:00 (16:00)	West Brom	2–2	Watford	24,606	The Hawthorns	Michael Oliver	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
7	Sat	2017-09-30	15:00 (16:00)	Stoke City	2–1	Southampton	29,285	Bet365 Stadium	Mike Jones	Match Report	
7	Sat	2017-09-30	15:00 (16:00)	Bournemouth	0–0	Leicester City	10,444	Vitality Stadium	Graham Scott	Match Report	
7	Sat	2017-09-30	17:30 (18:30)	Chelsea	0–1	Manchester City	41,530	Stamford Bridge	Martin Atkinson	Match Report	
7	Sun	2017-10-01	12:00 (13:00)	Arsenal	2–0	Brighton	59,378	Emirates Stadium	Kevin Friend	Match Report	
7	Sun	2017-10-01	14:15 (15:15)	Everton	0–1	Burnley	38,448	Goodison Park	Jonathan Moss	Match Report	
7	Sun	2017-10-01	16:30 (17:30)	Newcastle United	1–1	Liverpool	52,303	St. James' Park	Craig Pawson	Match Report	
8	Sat	2017-10-14	12:30 (13:30)	Liverpool	0–0	Manchester Utd	52,912	Anfield	Martin Atkinson	Match Report	
8	Sat	2017-10-14	15:00 (16:00)	Burnley	1–1	West Ham United	20,945	Turf Moor	Stuart Attwell	Match Report	
8	Sat	2017-10-14	15:00 (16:00)	Crystal Palace	2–1	Chelsea	25,480	Selhurst Park	Andre Marriner	Match Report	
8	Sat	2017-10-14	15:00 (16:00)	Tottenham Hotspur	1–0	Bournemouth	73,502	Wembley Stadium	Robert Madley	Match Report	
8	Sat	2017-10-14	15:00 (16:00)	Manchester City	7–2	Stoke City	54,128	Etihad Stadium	Craig Pawson	Match Report	
8	Sat	2017-10-14	15:00 (16:00)	Swansea City	2–0	Huddersfield Town	20,657	Liberty Stadium	Paul Tierney	Match Report	
8	Sat	2017-10-14	17:30 (18:30)	Watford	2–1	Arsenal	20,384	Vicarage Road Stadium	Niel Swarbrick	Match Report	
8	Sun	2017-10-15	13:30 (14:30)	Brighton	1–1	Everton	30,565	The American Express Community Stadium	Michael Oliver	Match Report	
8	Sun	2017-10-15	16:00 (17:00)	Southampton	2–2	Newcastle United	31,437	St. Mary's Stadium	Kevin Friend	Match Report	
8	Mon	2017-10-16	20:00 (21:00)	Leicester City	1–1	West Brom	30,203	King Power Stadium	Mike Dean	Match Report	
9	Fri	2017-10-20	20:00 (21:00)	West Ham United	0–3	Brighton	56,977	London Stadium	Martin Atkinson	Match Report	
9	Sat	2017-10-21	12:30 (13:30)	Chelsea	4–2	Watford	41,467	Stamford Bridge	Jonathan Moss	Match Report	
9	Sat	2017-10-21	15:00 (16:00)	Stoke City	1–2	Bournemouth	29,500	Bet365 Stadium	Lee Probert	Match Report	
9	Sat	2017-10-21	15:00 (16:00)	Swansea City	1–2	Leicester City	20,521	Liberty Stadium	Michael Oliver	Match Report	
9	Sat	2017-10-21	15:00 (16:00)	Manchester City	3–0	Burnley	54,118	Etihad Stadium	Roger East	Match Report	
9	Sat	2017-10-21	15:00 (16:00)	Huddersfield Town	2–1	Manchester Utd	24,426	The John Smith's Stadium	Lee Mason	Match Report	
9	Sat	2017-10-21	15:00 (16:00)	Newcastle United	1–0	Crystal Palace	52,251	St. James' Park	Stuart Attwell	Match Report	
9	Sat	2017-10-21	17:30 (18:30)	Southampton	1–0	West Brom	29,947	St. Mary's Stadium	Graham Scott	Match Report	
9	Sun	2017-10-22	13:30 (14:30)	Everton	2–5	Arsenal	39,189	Goodison Park	Craig Pawson	Match Report	
9	Sun	2017-10-22	16:00 (17:00)	Tottenham Hotspur	4–1	Liverpool	80,827	Wembley Stadium	Andre Marriner	Match Report	
10	Sat	2017-10-28	12:30 (13:30)	Manchester Utd	1–0	Tottenham Hotspur	75,034	Old Trafford	Jonathan Moss	Match Report	
10	Sat	2017-10-28	15:00 (16:00)	Arsenal	2–1	Swansea City	59,493	Emirates Stadium	Lee Mason	Match Report	
10	Sat	2017-10-28	15:00 (16:00)	Watford	0–1	Stoke City	20,087	Vicarage Road Stadium	Michael Oliver	Match Report	
10	Sat	2017-10-28	15:00 (16:00)	West Brom	2–3	Manchester City	24,003	The Hawthorns	Mike Jones	Match Report	
10	Sat	2017-10-28	15:00 (16:00)	Liverpool	3–0	Huddersfield Town	53,268	Anfield	Kevin Friend	Match Report	
10	Sat	2017-10-28	15:00 (16:00)	Crystal Palace	2–2	West Ham United	25,242	Selhurst Park	Robert Madley	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
10	Sat	2017-10-28	17:30 (18:30)	Bournemouth	0–1	Chelsea	10,998	Vitality Stadium	Craig Pawson	Match Report	
10	Sun	2017-10-29	13:30 (14:30)	Brighton	1–1	Southampton	30,564	The American Express Community Stadium	Niel Swarbrick	Match Report	
10	Sun	2017-10-29	16:00 (17:00)	Leicester City	2–0	Everton	31,891	King Power Stadium	Andre Marriner	Match Report	
10	Mon	2017-10-30	20:00 (21:00)	Burnley	1–0	Newcastle United	21,031	Turf Moor	Mike Dean	Match Report	
11	Sat	2017-11-04	12:30 (13:30)	Stoke City	2–2	Leicester City	29,602	Bet365 Stadium	Robert Madley	Match Report	
11	Sat	2017-11-04	15:00 (16:00)	Newcastle United	0–1	Bournemouth	52,237	St. James' Park	Paul Tierney	Match Report	
11	Sat	2017-11-04	15:00 (16:00)	Huddersfield Town	1–0	West Brom	24,169	The John Smith's Stadium	Roger East	Match Report	
11	Sat	2017-11-04	15:00 (16:00)	Southampton	0–1	Burnley	30,491	St. Mary's Stadium	Lee Probert	Match Report	
11	Sat	2017-11-04	15:00 (16:00)	Swansea City	0–1	Brighton	20,822	Liberty Stadium	Mike Dean	Match Report	
11	Sat	2017-11-04	17:30 (18:30)	West Ham United	1–4	Liverpool	56,961	London Stadium	Niel Swarbrick	Match Report	
11	Sun	2017-11-05	12:00 (13:00)	Tottenham Hotspur	1–0	Crystal Palace	65,270	Wembley Stadium	Kevin Friend	Match Report	
11	Sun	2017-11-05	14:15 (15:15)	Manchester City	3–1	Arsenal	54,286	Etihad Stadium	Michael Oliver	Match Report	
11	Sun	2017-11-05	16:30 (17:30)	Everton	3–2	Watford	38,609	Goodison Park	Graham Scott	Match Report	
11	Sun	2017-11-05	16:30 (17:30)	Chelsea	1–0	Manchester Utd	41,615	Stamford Bridge	Anthony Taylor	Match Report	
12	Sat	2017-11-18	12:30 (13:30)	Arsenal	2–0	Tottenham Hotspur	59,530	Emirates Stadium	Mike Dean	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	West Brom	0–4	Chelsea	23,592	The Hawthorns	Jonathan Moss	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	Burnley	2–0	Swansea City	18,895	Turf Moor	Martin Atkinson	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	Liverpool	3–0	Southampton	53,256	Anfield	Mike Jones	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	Crystal Palace	2–2	Everton	25,526	Selhurst Park	Anthony Taylor	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	Leicester City	0–2	Manchester City	31,908	King Power Stadium	Graham Scott	Match Report	
12	Sat	2017-11-18	15:00 (16:00)	Bournemouth	4–0	Huddersfield Town	10,879	Vitality Stadium	Lee Probert	Match Report	
12	Sat	2017-11-18	17:30 (18:30)	Manchester Utd	4–1	Newcastle United	75,035	Old Trafford	Craig Pawson	Match Report	
12	Sun	2017-11-19	16:00 (17:00)	Watford	2–0	West Ham United	20,018	Vicarage Road Stadium	Andre Marriner	Match Report	
12	Mon	2017-11-20	20:00 (21:00)	Brighton	2–2	Stoke City	29,676	The American Express Community Stadium	Lee Mason	Match Report	
13	Fri	2017-11-24	20:00 (21:00)	West Ham United	1–1	Leicester City	56,897	London Stadium	Martin Atkinson	Match Report	
13	Sat	2017-11-25	15:00 (16:00)	Newcastle United	0–3	Watford	52,188	St. James' Park	Chris Kavanagh	Match Report	
13	Sat	2017-11-25	15:00 (16:00)	Tottenham Hotspur	1–1	West Brom	65,905	Wembley Stadium	Mike Jones	Match Report	
13	Sat	2017-11-25	15:00 (16:00)	Manchester Utd	1–0	Brighton	75,018	Old Trafford	Niel Swarbrick	Match Report	
13	Sat	2017-11-25	15:00 (16:00)	Crystal Palace	2–1	Stoke City	23,723	Selhurst Park	Mike Dean	Match Report	
13	Sat	2017-11-25	15:00 (16:00)	Swansea City	0–0	Bournemouth	20,228	Liberty Stadium	Stuart Attwell	Match Report	
13	Sat	2017-11-25	17:30 (18:30)	Liverpool	1–1	Chelsea	53,225	Anfield	Michael Oliver	Match Report	
13	Sun	2017-11-26	13:30 (14:30)	Southampton	4–1	Everton	30,461	St. Mary's Stadium	Kevin Friend	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
13	Sun	2017-11-26	14:00 (15:00)	Burnley	0–1	Arsenal	21,722	Turf Moor	Lee Mason	Match Report	
13	Sun	2017-11-26	16:00 (17:00)	Huddersfield Town	1–2	Manchester City	24,121	The John Smith's Stadium	Craig Pawson	Match Report	
14	Tue	2017-11-28	19:45 (20:45)	Brighton	0–0	Crystal Palace	29,889	The American Express Community Stadium	Andre Marriner	Match Report	
14	Tue	2017-11-28	19:45 (20:45)	Leicester City	2–1	Tottenham Hotspur	31,950	King Power Stadium	Anthony Taylor	Match Report	
14	Tue	2017-11-28	20:00 (21:00)	Watford	2–4	Manchester Utd	20,552	Vicarage Road Stadium	Jonathan Moss	Match Report	
14	Tue	2017-11-28	20:00 (21:00)	West Brom	2–2	Newcastle United	25,534	The Hawthorns	Lee Probert	Match Report	
14	Wed	2017-11-29	19:45 (20:45)	Bournemouth	1–2	Burnley	10,302	Vitality Stadium	Roger East	Match Report	
14	Wed	2017-11-29	19:45 (20:45)	Arsenal	5–0	Huddersfield Town	59,285	Emirates Stadium	Graham Scott	Match Report	
14	Wed	2017-11-29	19:45 (20:45)	Chelsea	1–0	Swansea City	41,365	Stamford Bridge	Niel Swarbrick	Match Report	
14	Wed	2017-11-29	20:00 (21:00)	Manchester City	2–1	Southampton	53,407	Etihad Stadium	Paul Tierney	Match Report	
14	Wed	2017-11-29	20:00 (21:00)	Everton	4–0	West Ham United	38,242	Goodison Park	Michael Oliver	Match Report	
14	Wed	2017-11-29	20:00 (21:00)	Stoke City	0–3	Liverpool	29,423	Bet365 Stadium	Martin Atkinson	Match Report	
15	Sat	2017-12-02	12:30 (13:30)	Chelsea	3–1	Newcastle United	41,538	Stamford Bridge	Kevin Friend	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	Everton	2–0	Huddersfield Town	39,167	Goodison Park	Chris Kavanagh	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	West Brom	0–0	Crystal Palace	23,531	The Hawthorns	Michael Oliver	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	Brighton	1–5	Liverpool	30,634	The American Express Community Stadium	Graham Scott	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	Stoke City	2–1	Swansea City	28,261	Bet365 Stadium	Craig Pawson	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	Leicester City	1–0	Burnley	30,714	King Power Stadium	Paul Tierney	Match Report	
15	Sat	2017-12-02	15:00 (16:00)	Watford	1–1	Tottenham Hotspur	20,278	Vicarage Road Stadium	Martin Atkinson	Match Report	
15	Sat	2017-12-02	17:30 (18:30)	Arsenal	1–3	Manchester Utd	59,547	Emirates Stadium	Andre Marriner	Match Report	
15	Sun	2017-12-03	13:30 (14:30)	Bournemouth	1–1	Southampton	10,764	Vitality Stadium	Jonathan Moss	Match Report	
15	Sun	2017-12-03	16:00 (17:00)	Manchester City	2–1	West Ham United	54,203	Etihad Stadium	Mike Dean	Match Report	
16	Sat	2017-12-09	12:30 (13:30)	West Ham United	1–0	Chelsea	56,953	London Stadium	Anthony Taylor	Match Report	
16	Sat	2017-12-09	15:00 (16:00)	Tottenham Hotspur	5–1	Stoke City	62,202	Wembley Stadium	Roger East	Match Report	
16	Sat	2017-12-09	15:00 (16:00)	Huddersfield Town	2–0	Brighton	24,018	The John Smith's Stadium	Stuart Attwell	Match Report	
16	Sat	2017-12-09	15:00 (16:00)	Swansea City	1–0	West Brom	19,580	Liberty Stadium	Mike Dean	Match Report	
16	Sat	2017-12-09	15:00 (16:00)	Burnley	1–0	Watford	19,479	Turf Moor	Lee Probert	Match Report	
16	Sat	2017-12-09	15:00 (16:00)	Crystal Palace	2–2	Bournemouth	24,823	Selhurst Park	Kevin Friend	Match Report	
16	Sat	2017-12-09	17:30 (18:30)	Newcastle United	2–3	Leicester City	52,117	St. James' Park	Niel Swarbrick	Match Report	
16	Sun	2017-12-10	12:00 (13:00)	Southampton	1–1	Arsenal	31,643	St. Mary's Stadium	Robert Madley	Match Report	
16	Sun	2017-12-10	14:15 (15:15)	Liverpool	1–1	Everton	53,082	Anfield	Craig Pawson	Match Report	
16	Sun	2017-12-10	16:30 (17:30)	Manchester Utd	1–2	Manchester City	74,847	Old Trafford	Michael Oliver	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
17	Tue	2017-12-12	19:45 (20:45)	Burnley	1–0	Stoke City	19,909	Turf Moor	Mike Jones	Match Report	
17	Tue	2017-12-12	20:00 (21:00)	Huddersfield Town	1–3	Chelsea	24,169	The John Smith's Stadium	Andre Marriner	Match Report	
17	Tue	2017-12-12	20:00 (21:00)	Crystal Palace	2–1	Watford	23,566	Selhurst Park	Lee Mason	Match Report	
17	Wed	2017-12-13	19:45 (20:45)	Newcastle United	0–1	Everton	51,042	St. James' Park	Martin Atkinson	Match Report	
17	Wed	2017-12-13	19:45 (20:45)	Swansea City	0–4	Manchester City	20,870	Liberty Stadium	Anthony Taylor	Match Report	
17	Wed	2017-12-13	19:45 (20:45)	Southampton	1–4	Leicester City	27,714	St. Mary's Stadium	Chris Kavanagh	Match Report	
17	Wed	2017-12-13	20:00 (21:00)	Tottenham Hotspur	2–0	Brighton	46,438	Wembley Stadium	Robert Madley	Match Report	
17	Wed	2017-12-13	20:00 (21:00)	West Ham United	0–0	Arsenal	56,921	London Stadium	Jonathan Moss	Match Report	
17	Wed	2017-12-13	20:00 (21:00)	Liverpool	0–0	West Brom	53,243	Anfield	Paul Tierney	Match Report	
17	Wed	2017-12-13	20:00 (21:00)	Manchester Utd	1–0	Bournemouth	74,798	Old Trafford	Graham Scott	Match Report	
18	Sat	2017-12-16	12:30 (13:30)	Leicester City	0–3	Crystal Palace	31,081	King Power Stadium	Martin Atkinson	Match Report	
18	Sat	2017-12-16	15:00 (16:00)	Arsenal	1–0	Newcastle United	59,379	Emirates Stadium	Stuart Attwell	Match Report	
18	Sat	2017-12-16	15:00 (16:00)	Watford	1–4	Huddersfield Town	20,026	Vicarage Road Stadium	Michael Oliver	Match Report	
18	Sat	2017-12-16	15:00 (16:00)	Chelsea	1–0	Southampton	41,562	Stamford Bridge	Roger East	Match Report	
18	Sat	2017-12-16	15:00 (16:00)	Brighton	0–0	Burnley	29,921	The American Express Community Stadium	Chris Kavanagh	Match Report	
18	Sat	2017-12-16	16:00 (17:00)	Stoke City	0–3	West Ham United	29,265	Bet365 Stadium	Graham Scott	Match Report	
18	Sat	2017-12-16	17:30 (18:30)	Manchester City	4–1	Tottenham Hotspur	54,214	Etihad Stadium	Craig Pawson	Match Report	
18	Sun	2017-12-17	14:15 (15:15)	West Brom	1–2	Manchester Utd	24,782	The Hawthorns	Anthony Taylor	Match Report	
18	Sun	2017-12-17	16:30 (17:30)	Bournemouth	0–4	Liverpool	10,780	Vitality Stadium	Andre Marriner	Match Report	
18	Mon	2017-12-18	20:00 (21:00)	Everton	3–1	Swansea City	37,580	Goodison Park	Jonathan Moss	Match Report	
19	Fri	2017-12-22	19:45 (20:45)	Arsenal	3–3	Liverpool	59,409	Emirates Stadium	Martin Atkinson	Match Report	
19	Sat	2017-12-23	12:30 (13:30)	Everton	0–0	Chelsea	39,191	Goodison Park	Robert Madley	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	Brighton	1–0	Watford	30,473	The American Express Community Stadium	Paul Tierney	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	Swansea City	1–1	Crystal Palace	20,354	Liberty Stadium	Craig Pawson	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	Manchester City	4–0	Bournemouth	54,270	Etihad Stadium	Mike Jones	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	Stoke City	3–1	West Brom	29,057	Bet365 Stadium	Niel Swarbrick	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	West Ham United	2–3	Newcastle United	56,955	London Stadium	Lee Mason	Match Report	
19	Sat	2017-12-23	15:00 (16:00)	Southampton	1–1	Huddersfield Town	29,675	St. Mary's Stadium	Lee Probert	Match Report	
19	Sat	2017-12-23	17:30 (18:30)	Burnley	0–3	Tottenham Hotspur	21,650	Turf Moor	Michael Oliver	Match Report	
19	Sat	2017-12-23	19:45 (20:45)	Leicester City	2–2	Manchester Utd	32,202	King Power Stadium	Jonathan Moss	Match Report	
20	Tue	2017-12-26	12:30 (13:30)	Tottenham Hotspur	5–2	Southampton	55,412	Wembley Stadium	Graham Scott	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
20	Tue	2017-12-26	15:00 (16:00)	Chelsea	2–0	Brighton	41,568	Stamford Bridge	Mike Dean	Match Report	
20	Tue	2017-12-26	15:00 (16:00)	Huddersfield Town	1–1	Stoke City	24,047	The John Smith's Stadium	Anthony Taylor	Match Report	
20	Tue	2017-12-26	15:00 (16:00)	Watford	2–1	Leicester City	20,308	Vicarage Road Stadium	Chris Kavanagh	Match Report	
20	Tue	2017-12-26	15:00 (16:00)	Bournemouth	3–3	West Ham United	10,596	Vitality Stadium	Stuart Attwell	Match Report	
20	Tue	2017-12-26	15:00 (16:00)	Manchester Utd	2–2	Burnley	75,046	Old Trafford	Martin Atkinson	Match Report	
20	Tue	2017-12-26	15:00 (16:00)	West Brom	0–0	Everton	25,364	The Hawthorns	Roger East	Match Report	
20	Tue	2017-12-26	17:30 (18:30)	Liverpool	5–0	Swansea City	52,850	Anfield	Kevin Friend	Match Report	
20	Wed	2017-12-27	19:45 (20:45)	Newcastle United	0–1	Manchester City	52,311	St. James' Park	Andre Marriner	Match Report	
20	Thu	2017-12-28	20:00 (21:00)	Crystal Palace	2–3	Arsenal	25,762	Selhurst Park	Michael Oliver	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Newcastle United	0–0	Brighton	52,209	St. James' Park	Anthony Taylor	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Chelsea	5–0	Stoke City	41,433	Stamford Bridge	Kevin Friend	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Bournemouth	2–1	Everton	10,497	Vitality Stadium	Lee Probert	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Watford	1–2	Swansea City	20,002	Vicarage Road Stadium	Martin Atkinson	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Huddersfield Town	0–0	Burnley	24,095	The John Smith's Stadium	Paul Tierney	Match Report	
21	Sat	2017-12-30	15:00 (16:00)	Liverpool	2–1	Leicester City	53,226	Anfield	Niel Swarbrick	Match Report	
21	Sat	2017-12-30	17:30 (18:30)	Manchester Utd	0–0	Southampton	75,051	Old Trafford	Craig Pawson	Match Report	
21	Sun	2017-12-31	12:00 (13:00)	Crystal Palace	0–0	Manchester City	25,804	Selhurst Park	Jonathan Moss	Match Report	
21	Sun	2017-12-31	16:30 (17:30)	West Brom	1–1	Arsenal	26,223	The Hawthorns	Mike Dean	Match Report	
22	Mon	2018-01-01	12:30 (13:30)	Brighton	2–2	Bournemouth	30,152	The American Express Community Stadium	Michael Oliver	Match Report	
22	Mon	2018-01-01	15:00 (16:00)	Burnley	1–2	Liverpool	21,756	Turf Moor	Roger East	Match Report	
22	Mon	2018-01-01	15:00 (16:00)	Leicester City	3–0	Huddersfield Town	31,748	King Power Stadium	Graham Scott	Match Report	
22	Mon	2018-01-01	15:00 (16:00)	Stoke City	0–1	Newcastle United	28,471	Bet365 Stadium	Chris Kavanagh	Match Report	
22	Mon	2018-01-01	17:30 (18:30)	Everton	0–2	Manchester Utd	39,188	Goodison Park	Andre Marriner	Match Report	
22	Tue	2018-01-02	19:45 (20:45)	West Ham United	2–1	West Brom	56,888	London Stadium	Mike Jones	Match Report	
22	Tue	2018-01-02	19:45 (20:45)	Swansea City	0–2	Tottenham Hotspur	20,615	Liberty Stadium	Robert Madley	Match Report	
22	Tue	2018-01-02	19:45 (20:45)	Southampton	1–2	Crystal Palace	28,411	St. Mary's Stadium	Stuart Attwell	Match Report	
22	Tue	2018-01-02	20:00 (21:00)	Manchester City	3–1	Watford	53,556	Etihad Stadium	Lee Mason	Match Report	
22	Wed	2018-01-03	19:45 (20:45)	Arsenal	2–2	Chelsea	59,379	Emirates Stadium	Anthony Taylor	Match Report	
21	Thu	2018-01-04	20:00 (21:00)	Tottenham Hotspur	1–1	West Ham United	50,034	Wembley Stadium	Mike Dean	Match Report	
23	Sat	2018-01-13	15:00 (16:00)	Huddersfield Town	1–4	West Ham United	24,105	The John Smith's Stadium	Jonathan Moss	Match Report	
23	Sat	2018-01-13	15:00 (16:00)	Newcastle United	1–1	Swansea City	51,444	St. James' Park	Graham Scott	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
23	Sat	2018-01-13	15:00 (16:00)	Watford	2–2	Southampton	20,018	Vicarage Road Stadium	Roger East	Match Report	
23	Sat	2018-01-13	15:00 (16:00)	Crystal Palace	1–0	Burnley	24,696	Selhurst Park	Michael Oliver	Match Report	
23	Sat	2018-01-13	15:00 (16:00)	West Brom	2–0	Brighton	25,240	The Hawthorns	Martin Atkinson	Match Report	
23	Sat	2018-01-13	15:00 (16:00)	Chelsea	0–0	Leicester City	41,552	Stamford Bridge	Mike Jones	Match Report	
23	Sat	2018-01-13	17:30 (18:30)	Tottenham Hotspur	4–0	Everton	76,251	Wembley Stadium	Craig Pawson	Match Report	
23	Sun	2018-01-14	13:30 (14:30)	Bournemouth	2–1	Arsenal	10,836	Vitality Stadium	Kevin Friend	Match Report	
23	Sun	2018-01-14	16:00 (17:00)	Liverpool	4–3	Manchester City	53,285	Anfield	Andre Marriner	Match Report	
23	Mon	2018-01-15	20:00 (21:00)	Manchester Utd	3–0	Stoke City	74,726	Old Trafford	Anthony Taylor	Match Report	
24	Sat	2018-01-20	12:30 (13:30)	Brighton	0–4	Chelsea	30,600	The American Express Community Stadium	Jonathan Moss	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	Burnley	0–1	Manchester Utd	21,841	Turf Moor	Mike Dean	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	Everton	1–1	West Brom	39,061	Goodison Park	Stuart Attwell	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	West Ham United	1–1	Bournemouth	56,948	London Stadium	Martin Atkinson	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	Stoke City	2–0	Huddersfield Town	29,785	Bet365 Stadium	Michael Oliver	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	Arsenal	4–1	Crystal Palace	59,386	Emirates Stadium	Chris Kavanagh	Match Report	
24	Sat	2018-01-20	15:00 (16:00)	Leicester City	2–0	Watford	31,891	King Power Stadium	Lee Probert	Match Report	
24	Sat	2018-01-20	17:30 (18:30)	Manchester City	3–1	Newcastle United	54,452	Etihad Stadium	Paul Tierney	Match Report	
24	Sun	2018-01-21	16:00 (17:00)	Southampton	1–1	Tottenham Hotspur	31,361	St. Mary's Stadium	Kevin Friend	Match Report	
24	Mon	2018-01-22	20:00 (21:00)	Swansea City	1–0	Liverpool	20,886	Liberty Stadium	Niel Swarbrick	Match Report	
25	Tue	2018-01-30	19:45 (20:45)	West Ham United	1–1	Crystal Palace	56,911	London Stadium	Niel Swarbrick	Match Report	
25	Tue	2018-01-30	19:45 (20:45)	Swansea City	3–1	Arsenal	20,819	Liberty Stadium	Lee Mason	Match Report	
25	Tue	2018-01-30	20:00 (21:00)	Huddersfield Town	0–3	Liverpool	24,121	The John Smith's Stadium	Kevin Friend	Match Report	
25	Wed	2018-01-31	19:45 (20:45)	Chelsea	0–3	Bournemouth	41,464	Stamford Bridge	Lee Probert	Match Report	
25	Wed	2018-01-31	19:45 (20:45)	Everton	2–1	Leicester City	38,390	Goodison Park	Chris Kavanagh	Match Report	
25	Wed	2018-01-31	19:45 (20:45)	Southampton	1–1	Brighton	30,034	St. Mary's Stadium	Mike Dean	Match Report	
25	Wed	2018-01-31	19:45 (20:45)	Newcastle United	1–1	Burnley	50,174	St. James' Park	Simon Hooper	Match Report	
25	Wed	2018-01-31	20:00 (21:00)	Manchester City	3–0	West Brom	53,241	Etihad Stadium	Robert Madley	Match Report	
25	Wed	2018-01-31	20:00 (21:00)	Tottenham Hotspur	2–0	Manchester Utd	81,978	Wembley Stadium	Andre Marriner	Match Report	
25	Wed	2018-01-31	20:00 (21:00)	Stoke City	0–0	Watford	27,458	Bet365 Stadium	Jonathan Moss	Match Report	
26	Sat	2018-02-03	12:30 (13:30)	Burnley	1–1	Manchester City	21,658	Turf Moor	Martin Atkinson	Match Report	
26	Sat	2018-02-03	15:00 (16:00)	Leicester City	1–1	Swansea City	31,179	King Power Stadium	Anthony Taylor	Match Report	
26	Sat	2018-02-03	15:00 (16:00)	Bournemouth	2–1	Stoke City	10,614	Vitality Stadium	Paul Tierney	Match Report	
26	Sat	2018-02-03	15:00 (16:00)	West Brom	2–3	Southampton	25,911	The Hawthorns	Michael Oliver	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
26	Sat	2018-02-03	15:00 (16:00)	Manchester Utd	2–0	Huddersfield Town	74,742	Old Trafford	Stuart Attwell	Match Report	
26	Sat	2018-02-03	15:00 (16:00)	Brighton	3–1	West Ham United	30,589	The American Express Community Stadium	Roger East	Match Report	
26	Sat	2018-02-03	17:30 (18:30)	Arsenal	5–1	Everton	59,306	Emirates Stadium	Niel Swarbrick	Match Report	
26	Sun	2018-02-04	14:15 (15:15)	Crystal Palace	1–1	Newcastle United	25,746	Selhurst Park	Andre Marriner	Match Report	
26	Sun	2018-02-04	16:30 (17:30)	Liverpool	2–2	Tottenham Hotspur	53,213	Anfield	Jonathan Moss	Match Report	
26	Mon	2018-02-05	20:00 (21:00)	Watford	4–1	Chelsea	20,157	Vicarage Road Stadium	Mike Dean	Match Report	
27	Sat	2018-02-10	12:30 (13:30)	Tottenham Hotspur	1–0	Arsenal	83,222	Wembley Stadium	Anthony Taylor	Match Report	
27	Sat	2018-02-10	15:00 (16:00)	Swansea City	1–0	Burnley	20,179	Liberty Stadium	Andre Marriner	Match Report	
27	Sat	2018-02-10	15:00 (16:00)	West Ham United	2–0	Watford	56,197	London Stadium	Graham Scott	Match Report	
27	Sat	2018-02-10	15:00 (16:00)	Stoke City	1–1	Brighton	29,876	Bet365 Stadium	Robert Madley	Match Report	
27	Sat	2018-02-10	15:00 (16:00)	Everton	3–1	Crystal Palace	39,139	Goodison Park	Jonathan Moss	Match Report	
27	Sat	2018-02-10	17:30 (18:30)	Manchester City	5–1	Leicester City	54,416	Etihad Stadium	Mike Jones	Match Report	
27	Sun	2018-02-11	12:00 (13:00)	Huddersfield Town	4–1	Bournemouth	23,823	The John Smith's Stadium	Michael Oliver	Match Report	
27	Sun	2018-02-11	14:15 (15:15)	Newcastle United	1–0	Manchester Utd	52,309	St. James' Park	Craig Pawson	Match Report	
27	Sun	2018-02-11	16:30 (17:30)	Southampton	0–2	Liverpool	31,915	St. Mary's Stadium	Martin Atkinson	Match Report	
27	Mon	2018-02-12	20:00 (21:00)	Chelsea	3–0	West Brom	41,071	Stamford Bridge	Lee Mason	Match Report	
28	Sat	2018-02-24	12:30 (13:30)	Leicester City	1–1	Stoke City	31,769	King Power Stadium	Michael Oliver	Match Report	
28	Sat	2018-02-24	15:00 (16:00)	Bournemouth	2–2	Newcastle United	10,808	Vitality Stadium	Roger East	Match Report	
28	Sat	2018-02-24	15:00 (16:00)	Burnley	1–1	Southampton	20,982	Turf Moor	Robert Madley	Match Report	
28	Sat	2018-02-24	15:00 (16:00)	West Brom	1–2	Huddersfield Town	25,920	The Hawthorns	Jonathan Moss	Match Report	
28	Sat	2018-02-24	15:00 (16:00)	Liverpool	4–1	West Ham United	53,256	Anfield	Stuart Attwell	Match Report	
28	Sat	2018-02-24	15:00 (16:00)	Brighton	4–1	Swansea City	30,523	The American Express Community Stadium	Mike Dean	Match Report	
28	Sat	2018-02-24	17:30 (18:30)	Watford	1–0	Everton	20,430	Vicarage Road Stadium	Anthony Taylor	Match Report	
28	Sun	2018-02-25	12:00 (13:00)	Crystal Palace	0–1	Tottenham Hotspur	25,287	Selhurst Park	Kevin Friend	Match Report	
28	Sun	2018-02-25	14:05 (15:05)	Manchester Utd	2–1	Chelsea	75,060	Old Trafford	Martin Atkinson	Match Report	
28	Thu	2018-03-01	19:45 (20:45)	Arsenal	0–3	Manchester City	58,420	Emirates Stadium	Andre Marriner	Match Report	
29	Sat	2018-03-03	12:30 (13:30)	Burnley	2–1	Everton	20,802	Turf Moor	Chris Kavanagh	Match Report	
29	Sat	2018-03-03	15:00 (16:00)	Tottenham Hotspur	2–0	Huddersfield Town	68,311	Wembley Stadium	Kevin Friend	Match Report	
29	Sat	2018-03-03	15:00 (16:00)	Watford	1–0	West Brom	20,022	Vicarage Road Stadium	Paul Tierney	Match Report	
29	Sat	2018-03-03	15:00 (16:00)	Leicester City	1–1	Bournemouth	31,384	King Power Stadium	Lee Probert	Match Report	
29	Sat	2018-03-03	15:00 (16:00)	Southampton	0–0	Stoke City	30,335	St. Mary's Stadium	Anthony Taylor	Match Report	
29	Sat	2018-03-03	15:00 (16:00)	Swansea City	4–1	West Ham United	20,829	Liberty Stadium	Martin Atkinson	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
29	Sat	2018-03-03	17:30 (18:30)	Liverpool	2–0	Newcastle United	53,287	Anfield	Graham Scott	Match Report	
29	Sun	2018-03-04	13:30 (14:30)	Brighton	2–1	Arsenal	30,620	The American Express Community Stadium	Stuart Attwell	Match Report	
29	Sun	2018-03-04	16:00 (17:00)	Manchester City	1–0	Chelsea	54,328	Etihad Stadium	Michael Oliver	Match Report	
29	Mon	2018-03-05	20:00 (21:00)	Crystal Palace	2–3	Manchester Utd	25,840	Selhurst Park	Niel Swarbrick	Match Report	
30	Sat	2018-03-10	12:30 (13:30)	Manchester Utd	2–1	Liverpool	74,855	Old Trafford	Craig Pawson	Match Report	
30	Sat	2018-03-10	15:00 (16:00)	West Ham United	0–3	Burnley	56,904	London Stadium	Lee Mason	Match Report	
30	Sat	2018-03-10	15:00 (16:00)	Huddersfield Town	0–0	Swansea City	23,567	The John Smith's Stadium	Michael Oliver	Match Report	
30	Sat	2018-03-10	15:00 (16:00)	Newcastle United	3–0	Southampton	52,246	St. James' Park	Andre Marriner	Match Report	
30	Sat	2018-03-10	15:00 (16:00)	Everton	2–0	Brighton	39,199	Goodison Park	Roger East	Match Report	
30	Sat	2018-03-10	15:00 (16:00)	West Brom	1–4	Leicester City	23,558	The Hawthorns	Robert Madley	Match Report	
30	Sat	2018-03-10	17:30 (18:30)	Chelsea	2–1	Crystal Palace	40,800	Stamford Bridge	Anthony Taylor	Match Report	
30	Sun	2018-03-11	13:30 (14:30)	Arsenal	3–0	Watford	59,131	Emirates Stadium	Martin Atkinson	Match Report	
30	Sun	2018-03-11	16:00 (17:00)	Bournemouth	1–4	Tottenham Hotspur	10,623	Vitality Stadium	Mike Dean	Match Report	
30	Mon	2018-03-12	20:00 (21:00)	Stoke City	0–2	Manchester City	29,138	Bet365 Stadium	Jonathan Moss	Match Report	
31	Sat	2018-03-17	15:00 (16:00)	Stoke City	1–2	Everton	30,022	Bet365 Stadium	Martin Atkinson	Match Report	
31	Sat	2018-03-17	15:00 (16:00)	Bournemouth	2–1	West Brom	10,242	Vitality Stadium	Graham Scott	Match Report	
31	Sat	2018-03-17	15:00 (16:00)	Huddersfield Town	0–2	Crystal Palace	23,918	The John Smith's Stadium	Mike Dean	Match Report	
31	Sat	2018-03-17	17:30 (18:30)	Liverpool	5–0	Watford	53,287	Anfield	Anthony Taylor	Match Report	
32	Sat	2018-03-31	12:30 (13:30)	Crystal Palace	1–2	Liverpool	25,807	Selhurst Park	Niel Swarbrick	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	West Ham United	3–0	Southampton	56,882	London Stadium	Jonathan Moss	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	Watford	2–2	Bournemouth	20,393	Vicarage Road Stadium	Andy Madley	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	West Brom	1–2	Burnley	23,455	The Hawthorns	Lee Probert	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	Brighton	0–2	Leicester City	30,629	The American Express Community Stadium	Chris Kavanagh	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	Manchester Utd	2–0	Swansea City	75,038	Old Trafford	Robert Madley	Match Report	
32	Sat	2018-03-31	15:00 (16:00)	Newcastle United	1–0	Huddersfield Town	52,261	St. James' Park	Martin Atkinson	Match Report	
32	Sat	2018-03-31	17:30 (18:30)	Everton	1–3	Manchester City	39,221	Goodison Park	Paul Tierney	Match Report	
32	Sun	2018-04-01	13:30 (14:30)	Arsenal	3–0	Stoke City	59,371	Emirates Stadium	Craig Pawson	Match Report	
32	Sun	2018-04-01	16:00 (17:00)	Chelsea	1–3	Tottenham Hotspur	41,364	Stamford Bridge	Andre Marriner	Match Report	
33	Sat	2018-04-07	12:30 (13:30)	Everton	0–0	Liverpool	39,220	Goodison Park	Michael Oliver	Match Report	
33	Sat	2018-04-07	15:00 (16:00)	Brighton	1–1	Huddersfield Town	30,501	The American Express Community Stadium	Anthony Taylor	Match Report	
33	Sat	2018-04-07	15:00 (16:00)	Leicester City	1–2	Newcastle United	32,066	King Power Stadium	Stuart Attwell	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
33	Sat	2018-04-07	15:00 (16:00)	Watford	1–2	Burnley	20,044	Vicarage Road Stadium	Paul Tierney	Match Report	
33	Sat	2018-04-07	15:00 (16:00)	Bournemouth	2–2	Crystal Palace	10,730	Vitality Stadium	Jonathan Moss	Match Report	
33	Sat	2018-04-07	15:00 (16:00)	West Brom	1–1	Swansea City	23,297	The Hawthorns	Roger East	Match Report	
33	Sat	2018-04-07	15:00 (16:00)	Stoke City	1–2	Tottenham Hotspur	29,515	Bet365 Stadium	Graham Scott	Match Report	
33	Sat	2018-04-07	17:30 (18:30)	Manchester City	2–3	Manchester Utd	54,259	Etihad Stadium	Martin Atkinson	Match Report	
33	Sun	2018-04-08	14:15 (15:15)	Arsenal	3–2	Southampton	59,374	Emirates Stadium	Andre Marriner	Match Report	
33	Sun	2018-04-08	16:30 (17:30)	Chelsea	1–1	West Ham United	41,324	Stamford Bridge	Kevin Friend	Match Report	
34	Sat	2018-04-14	12:30 (13:30)	Southampton	2–3	Chelsea	31,764	St. Mary's Stadium	Mike Dean	Match Report	
34	Sat	2018-04-14	15:00 (16:00)	Crystal Palace	3–2	Brighton	24,656	Selhurst Park	Andre Marriner	Match Report	
34	Sat	2018-04-14	15:00 (16:00)	Burnley	2–1	Leicester City	21,727	Turf Moor	Martin Atkinson	Match Report	
34	Sat	2018-04-14	15:00 (16:00)	Huddersfield Town	1–0	Watford	23,961	The John Smith's Stadium	Craig Pawson	Match Report	
34	Sat	2018-04-14	15:00 (16:00)	Swansea City	1–1	Everton	20,933	Liberty Stadium	Lee Mason	Match Report	
34	Sat	2018-04-14	17:30 (18:30)	Liverpool	3–0	Bournemouth	52,959	Anfield	Chris Kavanagh	Match Report	
34	Sat	2018-04-14	19:45 (20:45)	Tottenham Hotspur	1–3	Manchester City	80,811	Wembley Stadium	Jonathan Moss	Match Report	
34	Sun	2018-04-15	13:30 (14:30)	Newcastle United	2–1	Arsenal	52,210	St. James' Park	Anthony Taylor	Match Report	
34	Sun	2018-04-15	16:00 (17:00)	Manchester Utd	0–1	West Brom	75,095	Old Trafford	Paul Tierney	Match Report	
34	Mon	2018-04-16	20:00 (21:00)	West Ham United	1–1	Stoke City	56,795	London Stadium	Michael Oliver	Match Report	
35	Tue	2018-04-17	19:45 (20:45)	Brighton	1–1	Tottenham Hotspur	30,440	The American Express Community Stadium	Kevin Friend	Match Report	
35	Wed	2018-04-18	19:45 (20:45)	Bournemouth	0–2	Manchester Utd	10,952	Vitality Stadium	Graham Scott	Match Report	
31	Thu	2018-04-19	19:45 (20:45)	Burnley	1–2	Chelsea	21,264	Turf Moor	Robert Madley	Match Report	
35	Thu	2018-04-19	19:45 (20:45)	Leicester City	0–0	Southampton	31,160	King Power Stadium	Roger East	Match Report	
35	Sat	2018-04-21	12:30 (13:30)	West Brom	2–2	Liverpool	24,520	The Hawthorns	Stuart Attwell	Match Report	
35	Sat	2018-04-21	15:00 (16:00)	Watford	0–0	Crystal Palace	20,401	Vicarage Road Stadium	Chris Kavanagh	Match Report	
35	Sun	2018-04-22	13:30 (14:30)	Stoke City	1–1	Burnley	29,532	Bet365 Stadium	Mike Dean	Match Report	
35	Sun	2018-04-22	13:30 (14:30)	Arsenal	4–1	West Ham United	59,422	Emirates Stadium	Lee Mason	Match Report	
35	Sun	2018-04-22	16:30 (17:30)	Manchester City	5–0	Swansea City	54,387	Etihad Stadium	Craig Pawson	Match Report	
35	Mon	2018-04-23	20:00 (21:00)	Everton	1–0	Newcastle United	39,061	Goodison Park	Robert Madley	Match Report	
36	Sat	2018-04-28	12:30 (13:30)	Liverpool	0–0	Stoke City	53,255	Anfield	Andre Marriner	Match Report	
36	Sat	2018-04-28	15:00 (16:00)	Newcastle United	0–1	West Brom	52,283	St. James' Park	David Coote	Match Report	
36	Sat	2018-04-28	15:00 (16:00)	Huddersfield Town	0–2	Everton	24,121	The John Smith's Stadium	Lee Probert	Match Report	
Wk	Day	Date	Time	Home	Score	Away	Attendance	Venue	Referee	Match Report	Notes
36	Sat	2018-04-28	15:00 (16:00)	Southampton	2–1	Bournemouth	31,778	St. Mary's Stadium	Anthony Taylor	Match Report	
36	Sat	2018-04-28	15:00 (16:00)	Burnley	0–0	Brighton	19,459	Turf Moor	Roger East	Match Report	
36	Sat	2018-04-28	15:00 (16:00)	Crystal Palace	5–0	Leicester City	25,750	Selhurst Park	Mike Dean	Match Report	
36	Sat	2018-04-28	17:30 (18:30)	Swansea City	0–1	Chelsea	20,900	Liberty Stadium	Jonathan Moss	Match Report	
36	Sun	2018-04-29	14:15 (15:15)	West Ham United	1–4	Manchester City	56,904	London Stadium	Niel Swarbrick	Match Report	
36	Sun	2018-04-29	16:30 (17:30)	Manchester Utd	2–1	Arsenal	75,035	Old Trafford	Kevin Friend	Match Report	
36	Mon	2018-04-30	20:00 (21:00)	Tottenham Hotspur	2–0	Watford	52,675	Wembley Stadium	Michael Oliver	Match Report	
37	Fri	2018-05-04	20:00 (21:00)	Brighton	1–0	Manchester Utd	30,611	The American Express Community Stadium	Craig Pawson	Match Report	
37	Sat	2018-05-05	12:30 (13:30)	Stoke City	1–2	Crystal Palace	29,687	Bet365 Stadium	Martin Atkinson	Match Report	
37	Sat	2018-05-05	15:00 (16:00)	Watford	2–1	Newcastle United	20,375	Vicarage Road Stadium	Roger East	Match Report	
37	Sat	2018-05-05	15:00 (16:00)	Leicester City	0–2	West Ham United	32,013	King Power Stadium	Chris Kavanagh	Match Report	
37	Sat	2018-05-05	15:00 (16:00)	West Brom	1–0	Tottenham Hotspur	23,685	The Hawthorns	Mike Jones	Match Report	
37	Sat	2018-05-05	15:00 (16:00)	Bournemouth	1–0	Swansea City	10,820	Vitality Stadium	Kevin Friend	Match Report	
37	Sat	2018-05-05	17:30 (18:30)	Everton	1–1	Southampton	38,225	Goodison Park	Jonathan Moss	Match Report	
37	Sun	2018-05-06	13:30 (14:30)	Manchester City	0–0	Huddersfield Town	54,350	Etihad Stadium	Mike Dean	Match Report	
37	Sun	2018-05-06	16:30 (17:30)	Chelsea	1–0	Liverpool	41,314	Stamford Bridge	Anthony Taylor	Match Report	
37	Sun	2018-05-06	16:30 (17:30)	Arsenal	5–0	Burnley	59,540	Emirates Stadium	Andre Marriner	Match Report	
31	Tue	2018-05-08	19:45 (20:45)	Swansea City	0–1	Southampton	20,858	Liberty Stadium	Michael Oliver	Match Report	
31	Wed	2018-05-09	19:45 (20:45)	Leicester City	3–1	Arsenal	32,095	King Power Stadium	Graham Scott	Match Report	
35	Wed	2018-05-09	19:45 (20:45)	Chelsea	1–1	Huddersfield Town	38,910	Stamford Bridge	Lee Mason	Match Report	
31	Wed	2018-05-09	20:00 (21:00)	Manchester City	3–1	Brighton	54,013	Etihad Stadium	Paul Tierney	Match Report	
31	Wed	2018-05-09	20:00 (21:00)	Tottenham Hotspur	1–0	Newcastle United	54,923	Wembley Stadium	Niel Swarbrick	Match Report	
31	Thu	2018-05-10	19:45 (20:45)	West Ham United	0–0	Manchester Utd	56,902	London Stadium	Jonathan Moss	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Swansea City	1–2	Stoke City	20,673	Liberty Stadium	Anthony Taylor	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Newcastle United	3–0	Chelsea	52,294	St. James' Park	Martin Atkinson	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Manchester Utd	1–0	Watford	75,049	Old Trafford	Lee Mason	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Liverpool	4–0	Brighton	50,752	Anfield	Kevin Friend	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Tottenham Hotspur	5–4	Leicester City	77,841	Wembley Stadium	Craig Pawson	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Crystal Palace	2–0	West Brom	25,357	Selhurst Park	Jonathan Moss	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Huddersfield Town	0–1	Arsenal	24,122	The John Smith's Stadium	Michael Oliver	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Southampton	0–1	Manchester City	31,882	St. Mary's Stadium	Andre Marriner	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	West Ham United	3–1	Everton	56,926	London Stadium	Graham Scott	Match Report	
38	Sun	2018-05-13	15:00 (16:00)	Burnley	1–2	Bournemouth	20,720	Turf Moor	Paul Tierney	Match Report	
"""

In [3]:
def parse_fbref_paste(raw_text, competition, season):
    rows = []
    for line in raw_text.strip().splitlines():
        parts = line.split("\t")
        if len(parts) < 7:
            continue
        # Skip header rows (FBRef repeats headers mid-page)
        if parts[0].strip() in ("Wk", "Round", ""):
            continue

        # Columns: Wk Day Date Time Home Score Away Attendance Venue Referee ...
        date_str  = parts[2].strip()
        home_team = parts[4].strip()
        score     = parts[5].strip()
        away_team = parts[6].strip()
        venue     = parts[8].strip() if len(parts) > 8 else ""
        round_    = parts[0].strip()  # week number or round name

        # Skip rows with no score (unplayed fixtures)
        if not any(c in score for c in ["\u2013", "-"]):
            continue
        if not date_str or not home_team or not away_team:
            continue

        # Parse score
        for sep in ["\u2013", "-"]:
            if sep in score:
                s = score.split(sep)
                try:
                    home_goals = int(s[0].strip())
                    away_goals = int(s[1].strip())
                    break
                except (ValueError, IndexError):
                    home_goals = away_goals = None
        else:
            home_goals = away_goals = None

        if home_goals is None:
            continue

        rows.append({
            "date":        date_str,
            "season":      season,
            "league_name": competition,
            "round":       round_,
            "home_team":   home_team,
            "away_team":   away_team,
            "home_goals":  home_goals,
            "away_goals":  away_goals,
            "venue":       venue,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df


parsed = parse_fbref_paste(RAW_TEXT, COMPETITION, SEASON)
print(f"Parsed {len(parsed)} fixtures for {COMPETITION} {SEASON}")
parsed.head(10)

Parsed 380 fixtures for Premier League 2017-2018


,date,season,league_name,round,home_team,away_team,home_goals,away_goals,venue
0,2017-08-11,2017-2018,Premier League,1,Arsenal,Leicester City,4,3,Emirates Stadium
1,2017-08-12,2017-2018,Premier League,1,Watford,Liverpool,3,3,Vicarage Road Stadium
2,2017-08-12,2017-2018,Premier League,1,Everton,Stoke City,1,0,Goodison Park
3,2017-08-12,2017-2018,Premier League,1,Chelsea,Burnley,2,3,Stamford Bridge
4,2017-08-12,2017-2018,Premier League,1,Crystal Palace,Huddersfield Town,0,3,Selhurst Park
5,2017-08-12,2017-2018,Premier League,1,West Brom,Bournemouth,1,0,The Hawthorns
6,2017-08-12,2017-2018,Premier League,1,Southampton,Swansea City,0,0,St. Mary's Stadium
7,2017-08-12,2017-2018,Premier League,1,Brighton,Manchester City,0,2,The American Express Community Stadium
8,2017-08-13,2017-2018,Premier League,1,Newcastle United,Tottenham Hotspur,0,2,St. James' Park
9,2017-08-13,2017-2018,Premier League,1,Manchester Utd,West Ham United,4,0,Old Trafford


In [4]:
# Append to master CSV (deduplicates on date + home_team + away_team)
if parsed.empty:
    print("Nothing to save — check the paste and re-run")
else:
    if MASTER_CSV.exists():
        existing = pd.read_csv(MASTER_CSV, parse_dates=["date"])
        combined = pd.concat([existing, parsed], ignore_index=True)
        combined = combined.drop_duplicates(subset=["date", "home_team", "away_team"])
    else:
        combined = parsed

    combined = combined.sort_values(["season", "league_name", "date"]).reset_index(drop=True)
    combined.to_csv(MASTER_CSV, index=False)
    print(f"Saved {len(combined)} total rows → {MASTER_CSV}")
    print()
    print(combined.groupby(["league_name", "season"]).size().unstack(fill_value=0).to_string())

Saved 380 total rows → /Users/admin/dev/algobetting/infra/data/json/fbref/fixtures_manual.csv

season          2017-2018
league_name              
Premier League        380
